# 📋 File 04 — Đánh giá mô hình (Evaluation)
**Brain Tumor Classification MRI Dataset**

File này đánh giá toàn diện model tốt nhất từ file 03 trên **Test set**:
- Accuracy, Precision, Recall, F1-Score theo từng class
- Confusion Matrix (số lượng + tỷ lệ %)
- ROC Curve và AUC Score
- Grad-CAM: xem model đang nhìn vào vùng nào của ảnh MRI
- Phân tích ảnh dự đoán sai
- Tổng kết toàn bộ dự án

⚠️ **Lưu ý:** Chạy file này SAU KHI đã chạy xong file 03!

## 📦 Phần A — Import thư viện

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from pathlib import Path

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)
from sklearn.preprocessing import label_binarize

plt.style.use('seaborn-v0_8-whitegrid')
print(f'✅ TensorFlow: {tf.__version__}')
print('✅ Import thư viện thành công!')

## ⚙️ Phần B — Load cấu hình và model tốt nhất từ file 03

In [ ]:
DATA_DIR  = Path('./data')
TEST_DIR  = DATA_DIR / 'Testing'

# [SỬA] Khớp với file 02 và 03
IMG_SIZE    = (240, 240)
BATCH_SIZE  = 32
RANDOM_SEED = 42
NUM_CLASSES = 4

CLASSES = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
CLASS_LABELS = {
    'glioma_tumor':     'Glioma',
    'meningioma_tumor': 'Meningioma',
    'no_tumor':         'Khong co u',
    'pituitary_tumor':  'Pituitary'
}
LABEL_NAMES = [CLASS_LABELS[c] for c in CLASSES]

df_test = pd.read_csv('df_test.csv')
print(f'✅ Test set: {len(df_test)} anh')
print(f'   Phan bo: {df_test["label"].value_counts().to_dict()}')

if os.path.exists('best_tl_model.h5'):
    model = keras.models.load_model('best_tl_model.h5')
    MODEL_NAME = 'EfficientNetB0 Transfer Learning'
    print(f'\n✅ Da load: best_tl_model.h5')
else:
    model = keras.models.load_model('best_cnn_model.h5')
    MODEL_NAME = 'CNN Baseline'
    print(f'\n✅ Da load: best_cnn_model.h5')

print(f'   Model: {MODEL_NAME}')


In [ ]:
# ============================================================
# Tạo test generator
# QUAN TRỌNG: shuffle=False để giữ đúng thứ tự so sánh nhãn
# ============================================================
test_datagen = ImageDataGenerator()  # KHÔNG rescale — EfficientNet tự normalize

test_generator = test_datagen.flow_from_dataframe(
    dataframe=df_test,
    x_col='filepath',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False           # KHONG shuffle — giữ đúng thứ tự
)

# Mapping index <-> tên class
CLASS_INDICES  = test_generator.class_indices
INDEX_TO_CLASS = {v: k for k, v in CLASS_INDICES.items()}
print(f'Class indices: {CLASS_INDICES}')

## 🔮 Phần C — Dự đoán toàn bộ Test Set

Đưa toàn bộ test set qua model để lấy:
- `y_pred_proba`: xác suất cho mỗi class (dùng cho ROC)
- `y_pred`: class có xác suất cao nhất (dùng cho Confusion Matrix, F1...)
- `y_true`: nhãn thật từ dataset

In [ ]:
# ============================================================
# Predict toàn bộ test set
# ============================================================
print('🔮 Dang du doan tren Test set...')

# y_pred_proba shape: (n_samples, 4) — xác suất mỗi class
y_pred_proba = model.predict(test_generator, verbose=1)

# Lấy class có xác suất cao nhất
y_pred = np.argmax(y_pred_proba, axis=1)

# Nhãn thật từ generator
y_true = test_generator.classes

# Chuyển index → tên class để in báo cáo
y_pred_names = [INDEX_TO_CLASS[i] for i in y_pred]
y_true_names = [INDEX_TO_CLASS[i] for i in y_true]

# Accuracy cơ bản
test_accuracy = np.mean(y_pred == y_true)
n_correct = np.sum(y_pred == y_true)
n_wrong   = np.sum(y_pred != y_true)

print(f'\n✅ Hoan tat du doan!')
print(f'   Dung : {n_correct} / {len(y_true)}')
print(f'   Sai  : {n_wrong}  / {len(y_true)}')
print(f'   Test Accuracy: {test_accuracy*100:.2f}%')

## 📊 Phần D — Classification Report chi tiết

**Giải thích từng chỉ số:**
- **Precision**: Trong số model nói là *Glioma*, bao nhiêu % thực sự là Glioma? → Đo mức độ "tin tưởng" của model
- **Recall (Sensitivity)**: Trong tất cả ảnh Glioma thật, model tìm được bao nhiêu %? → Quan trọng trong y tế (không bỏ sót bệnh)
- **F1-Score**: Trung bình hài hoà Precision và Recall → Chỉ số cân bằng nhất
- **Support**: Số ảnh thật của class đó trong test set

In [ ]:
# ============================================================
# In Classification Report
# ============================================================
report_dict = classification_report(
    y_true_names, y_pred_names,
    target_names=LABEL_NAMES,
    output_dict=True
)

print('=' * 65)
print(f'CLASSIFICATION REPORT — {MODEL_NAME}')
print('=' * 65)
print(classification_report(
    y_true_names, y_pred_names,
    target_names=LABEL_NAMES
))

In [ ]:
# ============================================================
# Vẽ biểu đồ Precision / Recall / F1 theo từng class
# ============================================================
metrics = {
    'Precision': [report_dict[l]['precision'] for l in LABEL_NAMES],
    'Recall':    [report_dict[l]['recall']    for l in LABEL_NAMES],
    'F1-Score':  [report_dict[l]['f1-score']  for l in LABEL_NAMES],
}

x     = np.arange(len(LABEL_NAMES))
width = 0.25
colors_bar = ['#2196F3', '#4CAF50', '#FF9800']

fig, ax = plt.subplots(figsize=(13, 6))
for i, (metric_name, values) in enumerate(metrics.items()):
    bars = ax.bar(x + i*width, values, width,
                  label=metric_name, color=colors_bar[i],
                  edgecolor='white', linewidth=1.2)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01,
                f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

ax.set_title(f'Precision / Recall / F1-Score theo tung Class\n{MODEL_NAME}',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Loai u nao', fontsize=12)
ax.set_ylabel('Diem so', fontsize=12)
ax.set_xticks(x + width)
ax.set_xticklabels(LABEL_NAMES, fontsize=11)
ax.set_ylim([0, 1.12])
ax.axhline(y=0.9, color='red', linestyle='--', alpha=0.5, linewidth=1, label='Nguong 90%')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('eval_metrics_by_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eval_metrics_by_class.png')

## 🔲 Phần E — Confusion Matrix

**Đọc ma trận:**
- Hàng = nhãn **thật**
- Cột = nhãn **dự đoán**
- **Đường chéo** (từ trái trên → phải dưới) = dự đoán **đúng**
- **Ngoài đường chéo** = dự đoán **sai** và hay nhầm với class nào

In [ ]:
# ============================================================
# Vẽ Confusion Matrix — 2 dạng: số lượng và tỷ lệ %
# ============================================================
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Confusion Matrix — {MODEL_NAME}',
             fontsize=14, fontweight='bold')

# --- Dạng 1: Số lượng tuyệt đối ---
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=LABEL_NAMES,
    yticklabels=LABEL_NAMES,
    linewidths=0.8, linecolor='white',
    annot_kws={'size': 13, 'weight': 'bold'},
    ax=axes[0]
)
axes[0].set_title('So luong anh', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Du doan', fontsize=11)
axes[0].set_ylabel('Thuc te', fontsize=11)
axes[0].tick_params(axis='x', rotation=20)
axes[0].tick_params(axis='y', rotation=0)

# --- Dạng 2: Tỷ lệ % theo hàng (= Recall mỗi class) ---
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(
    cm_pct, annot=True, fmt='.1f', cmap='Greens',
    xticklabels=LABEL_NAMES,
    yticklabels=LABEL_NAMES,
    linewidths=0.8, linecolor='white',
    annot_kws={'size': 12},
    ax=axes[1]
)
axes[1].set_title('Ty le % (moi hang = Recall)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Du doan', fontsize=11)
axes[1].set_ylabel('Thuc te', fontsize=11)
axes[1].tick_params(axis='x', rotation=20)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('eval_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Phân tích: class nào hay bị nhầm với class nào?
print('\nPhan tich nham lan chinh:')
for i, true_label in enumerate(LABEL_NAMES):
    row = cm[i].copy()
    row[i] = 0
    if row.max() > 0:
        confused_with = LABEL_NAMES[row.argmax()]
        print(f'  {true_label:<15} -> hay bi nham thanh "{confused_with}" ({row.max()} lan)')

## 📈 Phần F — ROC Curve & AUC

**ROC Curve** đo khả năng phân biệt đúng/sai của model ở mọi ngưỡng.

**AUC (Area Under Curve):**
- AUC = 1.0 → hoàn hảo
- AUC = 0.5 → không khá hơn đoán ngẫu nhiên
- AUC > 0.95 → xuất sắc cho bài toán y tế

In [ ]:
# ============================================================
# Vẽ ROC Curve theo phương pháp One-vs-Rest
# Mỗi class được coi là positive, còn lại là negative
# ============================================================

# Binarize nhãn thật: [0,1,2,3] → ma trận one-hot shape (n, 4)
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(9, 7))
colors_roc = ['#E91E63', '#2196F3', '#4CAF50', '#FF9800']
auc_scores = {}

for i, (label, color) in enumerate(zip(LABEL_NAMES, colors_roc)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
    auc_score   = auc(fpr, tpr)
    auc_scores[label] = auc_score
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f'{label}  (AUC = {auc_score:.4f})')

# Đường tham chiếu — random classifier
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.2, alpha=0.6,
        label='Random (AUC = 0.50)')

ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_xlabel('False Positive Rate  (1 - Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate  (Sensitivity / Recall)', fontsize=12)
ax.set_title(f'ROC Curve — One vs Rest\n{MODEL_NAME}',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('eval_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print('AUC Score tung class:')
for label, score in auc_scores.items():
    bar = '|' * int(score * 30)
    print(f'  {label:<15} AUC = {score:.4f}  {bar}')
print(f'\n  Mean AUC = {np.mean(list(auc_scores.values())):.4f}')

## 🌡️ Phần G — Grad-CAM: Model đang nhìn vào đâu?

**Grad-CAM (Gradient-weighted Class Activation Mapping)** tạo heatmap
chỉ ra vùng nào của ảnh MRI mà model tập trung khi dự đoán.

- **Vùng đỏ/vàng** = model chú ý nhiều nhất
- **Vùng xanh/tím** = model ít chú ý

Rất quan trọng để kiểm tra xem model có đang nhìn vào đúng khối u không!

In [ ]:
# ============================================================
# Hàm Grad-CAM
# ============================================================
def get_last_conv_layer(model):
    """Tìm tên lớp Conv2D cuối cùng trong model (kể cả nested model)."""
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
        if hasattr(layer, 'layers'):
            for sublayer in reversed(layer.layers):
                if isinstance(sublayer, tf.keras.layers.Conv2D):
                    return sublayer.name
    return None


def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    """
    Tạo Grad-CAM heatmap cho 1 ảnh.
    Trả về: (heatmap 2D, predicted_class_index)
    """
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index    = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads       = tape.gradient(class_channel, conv_outputs)
    pooled      = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap     = conv_outputs[0] @ pooled[..., tf.newaxis]
    heatmap     = tf.squeeze(heatmap)
    heatmap     = tf.maximum(heatmap, 0)
    heatmap     = heatmap / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)


def overlay_heatmap(img_bgr, heatmap, target_size, alpha=0.45):
    """Chồng heatmap lên ảnh gốc, trả về ảnh RGB."""
    img_rgb = cv2.cvtColor(cv2.resize(img_bgr, target_size), cv2.COLOR_BGR2RGB)
    heat    = cv2.resize(heatmap, target_size)
    heat    = np.uint8(255 * heat)
    heat_c  = cv2.applyColorMap(heat, cv2.COLORMAP_JET)
    heat_c  = cv2.cvtColor(heat_c, cv2.COLOR_BGR2RGB)
    blended = cv2.addWeighted(img_rgb, 1 - alpha, heat_c, alpha, 0)
    return img_rgb, blended


last_conv = get_last_conv_layer(model)
print(f'✅ Lop Conv cuoi: {last_conv}')

In [ ]:
# ============================================================
# Vẽ Grad-CAM — 1 ảnh dự đoán ĐÚNG mỗi class
# ============================================================
fig, axes = plt.subplots(len(CLASSES), 3, figsize=(13, len(CLASSES) * 4))
fig.suptitle('Grad-CAM — Vung model tap trung khi du doan',
             fontsize=14, fontweight='bold')

col_titles = ['Anh goc', 'Heatmap', 'Goc + Heatmap']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold', pad=10)

for row, cls in enumerate(CLASSES):
    # Tìm ảnh dự đoán đúng của class này
    cls_idx     = CLASS_INDICES[cls]
    correct_idx = np.where((y_true == cls_idx) & (y_pred == cls_idx))[0]

    if len(correct_idx) == 0:
        for col in range(3):
            axes[row, col].text(0.5, 0.5, 'Khong co anh\ndung',
                                ha='center', va='center', transform=axes[row, col].transAxes)
            axes[row, col].axis('off')
        continue

    sample_idx  = correct_idx[0]
    img_path    = df_test.iloc[sample_idx]['filepath']
    img_bgr     = cv2.imread(img_path)

    # Chuẩn bị ảnh cho model
    img_resized = cv2.resize(img_bgr, IMG_SIZE)
    img_arr     = img_resized.astype('float32')  # KHÔNG chia 255 — EfficientNet tự xử lý
    img_arr     = np.expand_dims(img_arr, axis=0)

    try:
        heatmap, pred_idx = make_gradcam_heatmap(img_arr, model, last_conv)
        orig_rgb, blended = overlay_heatmap(img_bgr, heatmap, IMG_SIZE)
        confidence = y_pred_proba[sample_idx][pred_idx] * 100

        axes[row, 0].imshow(orig_rgb)
        axes[row, 0].set_ylabel(CLASS_LABELS[cls], fontsize=12,
                                 fontweight='bold', rotation=0,
                                 labelpad=70, va='center')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(heatmap, cmap='jet')
        axes[row, 1].axis('off')

        axes[row, 2].imshow(blended)
        axes[row, 2].set_title(f'Du doan: {CLASS_LABELS[cls]}\n({confidence:.1f}%)',
                                fontsize=10, color='green')
        axes[row, 2].axis('off')

    except Exception as e:
        print(f'Loi Grad-CAM cho {cls}: {e}')
        for col in range(3):
            axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('eval_gradcam.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eval_gradcam.png')

## ❌ Phần H — Phân tích ảnh dự đoán SAI

Xem trực tiếp những ảnh model đoán sai để hiểu điểm yếu.
Từ đây có thể cải thiện model bằng cách thêm data augmentation
hoặc điều chỉnh kiến trúc.

In [ ]:
# ============================================================
# Hiển thị 8 ảnh dự đoán sai đầu tiên
# ============================================================
wrong_indices = np.where(y_pred != y_true)[0]
print(f'So anh du doan sai: {len(wrong_indices)} / {len(y_true)} '
      f'({len(wrong_indices)/len(y_true)*100:.1f}%)')

N_SHOW = min(8, len(wrong_indices))

if N_SHOW == 0:
    print('Model du doan dung het! Khong co anh sai.')
else:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f'Anh du doan SAI — {MODEL_NAME}',
                 fontsize=13, fontweight='bold', color='red')
    axes = axes.flatten()

    for i, idx in enumerate(wrong_indices[:N_SHOW]):
        img_path   = df_test.iloc[idx]['filepath']
        img        = cv2.imread(img_path)
        img        = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img        = cv2.resize(img, (200, 200))

        true_label = CLASS_LABELS[INDEX_TO_CLASS[y_true[idx]]]
        pred_label = CLASS_LABELS[INDEX_TO_CLASS[y_pred[idx]]]
        conf       = y_pred_proba[idx][y_pred[idx]] * 100

        axes[i].imshow(img)
        axes[i].set_title(
            f'Thuc te: {true_label}\nDoan sai: {pred_label}\nTin tuong: {conf:.1f}%',
            fontsize=9, color='red'
        )
        axes[i].axis('off')

    for j in range(N_SHOW, 8):
        axes[j].axis('off')

    plt.tight_layout()
    plt.savefig('eval_wrong_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: eval_wrong_predictions.png')

## ✅ Phần I — Tổng kết toàn bộ dự án

In [ ]:
# ============================================================
# Vẽ dashboard tổng kết dạng ảnh
# ============================================================
macro_p   = report_dict['macro avg']['precision']
macro_r   = report_dict['macro avg']['recall']
macro_f1  = report_dict['macro avg']['f1-score']
mean_auc  = np.mean(list(auc_scores.values()))

summary_metrics = {
    'Test\nAccuracy': test_accuracy,
    'Macro\nPrecision': macro_p,
    'Macro\nRecall': macro_r,
    'Macro\nF1-Score': macro_f1,
    'Mean\nAUC': mean_auc
}

fig, ax = plt.subplots(figsize=(13, 5))
bar_colors = ['#4CAF50' if v >= 0.9 else '#FF9800' if v >= 0.8 else '#F44336'
              for v in summary_metrics.values()]

bars = ax.bar(summary_metrics.keys(),
              [v * 100 for v in summary_metrics.values()],
              color=bar_colors, edgecolor='white', linewidth=2, width=0.5)

for bar, (k, v) in zip(bars, summary_metrics.items()):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.8,
            f'{v*100:.2f}%', ha='center', fontsize=13, fontweight='bold')

ax.axhline(y=90, color='green', linestyle='--', alpha=0.6, linewidth=1.5, label='Nguong 90%')
ax.axhline(y=80, color='orange', linestyle='--', alpha=0.5, linewidth=1, label='Nguong 80%')
ax.set_ylim([0, 110])
ax.set_ylabel('Diem so (%)', fontsize=12)
ax.set_title(f'Dashboard Ket Qua Tong The — {MODEL_NAME}',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('eval_summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# In báo cáo tổng kết cuối cùng
# ============================================================
print('=' * 62)
print('  BAO CAO TONG KET DU AN')
print('  Brain Tumor Classification MRI')
print('=' * 62)
print(f'  Model         : {MODEL_NAME}')
print(f'  Test set      : {len(y_true)} anh')
print(f'  Du doan dung  : {n_correct} anh')
print(f'  Du doan sai   : {n_wrong} anh')
print()
print('  Chi so tong the:')
print(f'    Test Accuracy    : {test_accuracy*100:.2f}%')
print(f'    Macro Precision  : {macro_p*100:.2f}%')
print(f'    Macro Recall     : {macro_r*100:.2f}%')
print(f'    Macro F1-Score   : {macro_f1*100:.2f}%')
print(f'    Mean AUC         : {mean_auc:.4f}')
print()
print('  F1-Score tung class:')
for label in LABEL_NAMES:
    f1  = report_dict[label]['f1-score']
    acc = report_dict[label]['recall']
    bar = '|' * int(f1 * 25)
    print(f'    {label:<15} F1={f1:.3f}  Recall={acc:.3f}  {bar}')
print()
print('  File da luu:')
files = [
    'eval_metrics_by_class.png  — Bieu do Precision/Recall/F1',
    'eval_confusion_matrix.png  — Ma tran nham lan',
    'eval_roc_curve.png         — Duong cong ROC va AUC',
    'eval_gradcam.png           — Grad-CAM heatmap',
    'eval_wrong_predictions.png — Anh du doan sai',
    'eval_summary_dashboard.png — Dashboard tong ket',
]
for f in files:
    print(f'    {f}')
print()
print('  HOAN TAT TOAN BO PIPELINE!')
print('  01_EDA → 02_Preprocessing → 03_Model → 04_Evaluation')
print('=' * 62)